<a href="https://colab.research.google.com/github/Roumyajit/Data-Science-Projects/blob/main/ipl_advanced_analytics_win_probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏏 IPL Advanced Analytics — Win Probability & Player Performance (ML + Deep Learning)

**Author:** Roumyajit Sarkar  
**Domain:** Sports Analytics  
**Level:** Advanced (ML + NLP + Deep Learning)

## Objective
Build a live IPL win probability predictor, player performance rating system, and match outcome predictor.

## Key Techniques
- Advanced Feature Engineering (Cricket Metrics)
- Live Win Probability using XGBoost (over-by-over)
- Player Performance Rating (Composite Score)
- Neural Network for Score Prediction
- Interactive Visualizations with Plotly

In [1]:
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn xgboost lightgbm tensorflow

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
import xgboost as xgb
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras import layers, models

print('✅ Libraries imported!')

✅ Libraries imported!


In [5]:
pip install kaggle

In [10]:
# Import the `userdata` module to securely access your Kaggle API credentials.
from google.colab import userdata
import os

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Write the Kaggle API credentials to kaggle.json
# The userdata.get function retrieves the secret values you set.
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"' + userdata.get('KAGGLE_USERNAME') + '","key":"' + userdata.get('KAGGLE_KEY') + '"}')

# Set permissions for the kaggle.json file
!chmod 600 ~/.kaggle/kaggle.json

print('Kaggle API configured.')

Kaggle API configured.


In [11]:
# Download the dataset
# The dataset URL was provided in the original notebook cell comment.
!kaggle datasets download -d patrickb1912/ipl-complete-dataset-20082020

print('Dataset downloaded.')

Dataset URL: https://www.kaggle.com/datasets/patrickb1912/ipl-complete-dataset-20082020
License(s): DbCL-1.0
100% 1.82M/1.82M [00:00<00:00, 170MB/s]

Dataset downloaded.


In [12]:
# Unzip the dataset files
!unzip -o ipl-complete-dataset-20082020.zip

print('Dataset unzipped and ready to use.')

Archive:  ipl-complete-dataset-20082020.zip
  inflating: deliveries.csv          
  inflating: matches.csv             
Dataset unzipped and ready to use.


## 📥 Step 1: Load IPL Data

In [13]:
# Dataset: IPL Complete Dataset from Kaggle
# https://www.kaggle.com/patrickb1912/ipl-complete-dataset-20082020
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print(f'Matches: {matches.shape}')
print(f'Deliveries: {deliveries.shape}')
print(f'\nIPL Seasons: {matches["season"].nunique()}')
print(f'Total Matches: {len(matches)}')
matches.head()

Matches: (1095, 20)
Deliveries: (260920, 17)

IPL Seasons: 17
Total Matches: 1095


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [14]:
# Dataset: IPL Complete Dataset from Kaggle
# https://www.kaggle.com/patrickb1912/ipl-complete-dataset-20082020
# The files are now unzipped in the current directory
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print(f'Matches: {matches.shape}')
print(f'Deliveries: {deliveries.shape}')
print(f'\nIPL Seasons: {matches["season"].nunique()}')
print(f'Total Matches: {len(matches)}')
matches.head()

Matches: (1095, 20)
Deliveries: (260920, 17)

IPL Seasons: 17
Total Matches: 1095


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [15]:
# === Advanced EDA ===

# Most successful teams
team_wins = matches['winner'].value_counts().reset_index()
team_wins.columns = ['team', 'wins']

fig = px.bar(team_wins, x='team', y='wins', color='wins',
             title='IPL Team Win Count (All Seasons)',
             color_continuous_scale='viridis')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Toss decision analysis
toss_win = matches[matches['toss_winner'] == matches['winner']]
toss_win_pct = len(toss_win) / len(matches) * 100
print(f'\nToss winner also won match: {toss_win_pct:.1f}% of the time')

# Toss decision
fig2 = px.pie(matches, names='toss_decision',
              title='Toss Decision: Bat vs Field')
fig2.show()


Toss winner also won match: 50.6% of the time


## ⚙️ Step 2: Feature Engineering

In [19]:
import numpy as np

# === PLAYER PERFORMANCE RATING ===

# Clean column names by stripping whitespace
deliveries.columns = deliveries.columns.str.strip()
print(f"Columns after stripping: {deliveries.columns.tolist()}")

# Batting stats
batting = deliveries.groupby('batter').agg(
    runs=('batsman_runs', 'sum'),
    balls=('ball', 'count'),
    fours=('batsman_runs', lambda x: (x == 4).sum()),
    sixes=('batsman_runs', lambda x: (x == 6).sum()),
    innings=('match_id', 'nunique')
).reset_index()

batting['avg'] = batting['runs'] / batting['innings']
batting['strike_rate'] = (batting['runs'] / batting['balls']) * 100
batting['boundary_pct'] = (batting['fours'] * 4 + batting['sixes'] * 6) / batting['runs'] * 100

# Bowling stats
bowling = deliveries.groupby('bowler').agg(
    wickets=('player_dismissed', lambda x: x.notna().sum()),
    runs_given=('total_runs', 'sum'),
    balls_bowled=('ball', 'count'),
    matches=('match_id', 'nunique')
).reset_index()

bowling['economy'] = (bowling['runs_given'] / bowling['balls_bowled']) * 6
bowling['avg'] = bowling['runs_given'] / bowling['wickets'].replace(0, np.nan)
bowling['wickets_per_match'] = bowling['wickets'] / bowling['matches']

print('Batting stats for top 10 batsmen:')
print(batting.sort_values('runs', ascending=False).head(10)[['batter','runs','avg','strike_rate']].to_string())

Columns after stripping: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']
Batting stats for top 10 batsmen:
             batter  runs        avg  strike_rate
631         V Kohli  8014  32.844262   128.511867
512        S Dhawan  6769  30.628959   123.454313
477       RG Sharma  6630  26.414343   127.918194
147       DA Warner  6567  35.690217   135.429986
546        SK Raina  5536  27.680000   132.535312
374        MS Dhoni  5243  22.995614   132.835065
30   AB de Villiers  5181  30.476471   148.580442
124        CH Gayle  4997  35.439716   142.121729
501      RV Uthappa  4954  25.147208   126.152279
282      KD Karthik  4843  20.785408   131.353404


In [20]:
# === WIN PROBABILITY FEATURE ENGINEERING ===
# For second innings chase scenarios

# Merge deliveries with match info
merged = deliveries.merge(matches[['id', 'winner', 'team1', 'team2']],
                          left_on='match_id', right_on='id')

# Get 2nd innings data only
second_innings = merged[merged['inning'] == 2].copy()

# Cumulative features over-by-over
second_innings = second_innings.sort_values(['match_id', 'over', 'ball'])

# Calculate cumulative runs, wickets per over
over_data = second_innings.groupby(['match_id', 'over']).agg(
    runs_in_over=('total_runs', 'sum'),
    wickets_in_over=('player_dismissed', lambda x: x.notna().sum()),
    batting_team=('batting_team', 'first'),
    bowling_team=('bowling_team', 'first'),
    winner=('winner', 'first')
).reset_index()

over_data['cumulative_runs'] = over_data.groupby('match_id')['runs_in_over'].cumsum()
over_data['cumulative_wickets'] = over_data.groupby('match_id')['wickets_in_over'].cumsum()
over_data['target_win'] = (over_data['batting_team'] == over_data['winner']).astype(int)

# Get target score from match data
target_scores = deliveries[deliveries['inning'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
target_scores.columns = ['match_id', 'target']
target_scores['target'] += 1  # +1 to win

over_data = over_data.merge(target_scores, on='match_id', how='left')
over_data['runs_needed'] = over_data['target'] - over_data['cumulative_runs']
over_data['overs_left'] = 20 - over_data['over']
over_data['required_run_rate'] = over_data['runs_needed'] / (over_data['overs_left'] + 0.001)
over_data['current_run_rate'] = over_data['cumulative_runs'] / (over_data['over'] + 1)
over_data['wickets_left'] = 10 - over_data['cumulative_wickets']

print(f'Win probability features ready! Shape: {over_data.shape}')

Win probability features ready! Shape: (20467, 16)


## 🤖 Step 3: Win Probability Model

In [21]:
features = ['over', 'cumulative_runs', 'cumulative_wickets', 'runs_needed',
            'overs_left', 'required_run_rate', 'current_run_rate', 'wickets_left']

X = over_data[features].dropna()
y = over_data.loc[X.index, 'target_win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost for win probability
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    use_label_encoder=False, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print('=== Win Probability Model ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC : {roc_auc_score if False else __import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, y_pred, target_names=['Lost', 'Won']))

=== Win Probability Model ===
Accuracy: 0.7938
ROC-AUC : 0.8910
              precision    recall  f1-score   support

        Lost       0.80      0.76      0.78      1976
         Won       0.79      0.83      0.81      2118

    accuracy                           0.79      4094
   macro avg       0.79      0.79      0.79      4094
weighted avg       0.79      0.79      0.79      4094



In [22]:
# Visualize Win Probability for a specific match
sample_match = over_data[over_data['match_id'] == over_data['match_id'].iloc[100]].copy()
sample_X = sample_match[features].dropna()
sample_match.loc[sample_X.index, 'win_prob'] = xgb_model.predict_proba(sample_X)[:, 1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sample_match['over'], y=sample_match['win_prob'],
    mode='lines+markers', name='Win Probability',
    line=dict(color='royalblue', width=3),
    fill='tozeroy', fillcolor='rgba(65, 105, 225, 0.2)'
))
fig.add_hline(y=0.5, line_dash='dash', line_color='red', annotation_text='50% Line')
fig.update_layout(
    title='IPL Live Win Probability (Over by Over)',
    xaxis_title='Over', yaxis_title='Win Probability',
    yaxis=dict(range=[0, 1]), height=450
)
fig.show()

print('\n✅ IPL Advanced Analytics Complete!')


✅ IPL Advanced Analytics Complete!
